# 11 Change Data Feed

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Enable Delta's <b>change data feed (CDF)</b>, read the row-level changes between versions (inserts, updates with before and after images, deletes), and use them to update a downstream table <b>incrementally</b> instead of recomputing it.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Recomputing a Gold table from a whole Silver table every hour gets expensive as data grows. With CDF, a job reads only what changed since its last run. CDF also feeds downstream systems (search indexes, caches, other databases) with exact changes, and it's an audit trail of who changed what. It appears in the exam's ingestion and transformation domains.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What CDF records</b><br>
With <code>delta.enableChangeDataFeed = true</code>, each commit that changes data also records the changed rows. Reading the feed returns the table's columns plus:<br>| Column | Meaning |<br>|---|---|<br>| <code>_change_type</code> | <code>insert</code>, <code>update_preimage</code> (row before), <code>update_postimage</code> (row after), <code>delete</code> |<br>| <code>_commit_version</code> | The version of the change |<br>| <code>_commit_timestamp</code> | When it was committed |
<ul><li>Only changes <b>after</b> CDF is enabled are recorded</li><li>Changes are kept with the same retention as data files (they are cleaned up by <code>VACUUM</code>)</li><li>Read with <code>readChangeFeed</code> (DataFrame, batch or streaming) or <code>table_changes()</code> (SQL)</li></ul>
</div>

```
version 1: INSERT (1, 'Amal', 100)            -> insert           (1, Amal, 100)
version 2: UPDATE amount = 150 WHERE id = 1   -> update_preimage  (1, Amal, 100)
                                                 update_postimage (1, Amal, 150)
version 3: DELETE WHERE id = 1                -> delete           (1, Amal, 150)
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A Silver <code>orders</code> table of 2 billion rows receives about 300,000 inserts and updates per hour. The Gold <code>daily_revenue</code> table used to be rebuilt from all of Silver every hour, taking 40 minutes. With CDF, the job reads only the last hour's changes, recomputes only the affected days, and <code>MERGE</code>s them into Gold in 2 minutes.
</div>

## Syntax

```sql
CREATE TABLE t (...) TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE t SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

SELECT * FROM table_changes('cat.sch.t', 2);        -- from version 2 to the latest
SELECT * FROM table_changes('cat.sch.t', 2, 5);     -- versions 2 to 5
```

```python
(spark.read.option("readChangeFeed", "true")
      .option("startingVersion", 2)            # or startingTimestamp
      .option("endingVersion", 5)              # optional
      .table("cat.sch.t"))
```

## Setup: a Silver orders table with CDF enabled

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an <code>orders</code> table with CDF on, then makes an insert, an update and a delete.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The history shows versions 0 (create), 1 (insert), 2 (update) and 3 (delete).
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
from pyspark.sql import functions as F

orders = f"{DB}.orders_cdf"
spark.sql(f"""
    CREATE OR REPLACE TABLE {orders} (order_id INT, customer STRING, order_date DATE, amount DOUBLE)
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
spark.sql(f"""INSERT INTO {orders} VALUES
    (1, 'Amal', DATE'2024-11-01', 120.0), (2, 'John', DATE'2024-11-01', 80.0),
    (3, 'Vivek', DATE'2024-11-02', 300.0), (4, 'Sara', DATE'2024-11-02', 45.0)""")   # v1
spark.sql(f"UPDATE {orders} SET amount = 95.0 WHERE order_id = 2")                     # v2
spark.sql(f"DELETE FROM {orders} WHERE order_id = 4")                                  # v3
spark.sql(f"DESCRIBE HISTORY {orders}").select("version", "operation").show()

## 1. Read all changes

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the change feed from version 1 with SQL <code>table_changes</code>, and the same with the DataFrame API.<br><br>
<b>Why it matters</b><br>Each change appears as one or two rows: inserts and deletes once, updates twice (before and after). That's enough to replay or audit every change.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 <code>insert</code> rows at version 1, an <code>update_preimage</code> (80.0) and <code>update_postimage</code> (95.0) for order 2 at version 2, and a <code>delete</code> for order 4 at version 3.
</div>

In [0]:
spark.sql(f"SELECT * FROM table_changes('{orders}', 1) ORDER BY _commit_version, order_id, _change_type").show()

changes = (spark.read.option("readChangeFeed", "true").option("startingVersion", 1).table(orders))
changes.groupBy("_commit_version", "_change_type").count().orderBy("_commit_version", "_change_type").show()

## 2. Read a range of versions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads only versions 2 to 3, as an incremental job would after having processed version 1.<br><br>
<b>Why it matters</b><br>Incremental jobs remember the <b>last version they processed</b> and ask for changes after it. That keeps each run small, whatever the size of the table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
3 rows: the update pre- and post-images and the delete.
</div>

In [0]:
spark.read.option("readChangeFeed", "true").option("startingVersion", 2).option("endingVersion", 3).table(orders) \
    .select("order_id", "amount", "_change_type", "_commit_version").show()

## 3. Use CDF to maintain a Gold table incrementally

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds <code>daily_revenue</code> once from the full Silver table, then processes <b>only the new changes</b> to update it, and checks that the result equals a full recompute.<br><br>
<b>Why it matters</b><br>This is the main use of CDF. The pattern:
<ol><li>Read changes since the last processed version</li><li>Find the affected keys (here, order dates)</li><li>Recompute those keys from Silver (or apply the deltas)</li><li><code>MERGE</code> the results into Gold, and store the new last-processed version</li></ol>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After new changes (a new order on 2024-11-03 and an update on 2024-11-01), only 2 dates are recomputed, and the incrementally maintained Gold table matches a full recompute exactly.
</div>

In [0]:
gold = f"{DB}.daily_revenue_cdf"

def full_recompute():
    return spark.table(orders).groupBy("order_date").agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders"))

full_recompute().write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold)
last_processed = spark.sql(f"DESCRIBE HISTORY {orders} LIMIT 1").first()["version"]
print("Gold built up to Silver version", last_processed)

# New activity in Silver
spark.sql(f"INSERT INTO {orders} VALUES (5, 'Priya', DATE'2024-11-03', 60.0)")
spark.sql(f"UPDATE {orders} SET amount = 130.0 WHERE order_id = 1")

# 1-2. Changes since the last processed version, and the affected dates
new_changes = spark.read.option("readChangeFeed", "true").option("startingVersion", last_processed + 1).table(orders)
affected = new_changes.select("order_date").distinct()
print("affected dates:", [r["order_date"] for r in affected.collect()])

# 3. Recompute only those dates from Silver
recomputed = (spark.table(orders).join(affected, "order_date")
              .groupBy("order_date").agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders")))
recomputed.createOrReplaceTempView("recomputed")

# 4. Merge into Gold (a date that lost all its orders would need a delete clause as well)
spark.sql(f"""
    MERGE INTO {gold} g USING recomputed r ON g.order_date = r.order_date
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
last_processed = spark.sql(f"DESCRIBE HISTORY {orders} LIMIT 1").first()["version"]

spark.table(gold).orderBy("order_date").show()
print("incremental == full recompute:", sorted(spark.table(gold).collect()) == sorted(full_recompute().collect()))

## 4. Net changes: the latest state per key

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Collapses a range of changes into one row per order: its final state, or a delete.<br><br>
<b>Why it matters</b><br>Downstream systems often only need the <b>net</b> effect of a batch of changes. Taking the latest change per key (ignoring pre-images) is the standard way to turn a change feed into an upsert or delete batch for <code>MERGE</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row per changed order with its latest <code>_change_type</code> and values.
</div>

In [0]:
from pyspark.sql import Window

net = (
    spark.read.option("readChangeFeed", "true").option("startingVersion", 1).table(orders)
    .filter("_change_type != 'update_preimage'")
    .withColumn("rn", F.row_number().over(Window.partitionBy("order_id").orderBy(F.desc("_commit_version"))))
    .filter("rn = 1").drop("rn")
)
net.select("order_id", "amount", "_change_type", "_commit_version").orderBy("order_id").show()

## 5. Streaming from the change feed (preview)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows how a streaming query reads the change feed continuously.<br><br>
<b>Why it matters</b><br>With <code>readStream</code> and a checkpoint, the stream remembers the last version processed by itself, which is the production-grade way to propagate changes. Structured Streaming is covered in <code>08_streaming_and_ingestion</code>.
</div>

```python
(spark.readStream
      .option("readChangeFeed", "true")
      .table("cat.sch.orders")
      .writeStream
      .option("checkpointLocation", "/Volumes/.../checkpoints/orders_to_gold")
      .foreachBatch(merge_changes_into_gold)        # a function that MERGEs each micro-batch
      .trigger(availableNow=True)
      .start())
```

## Under the hood

```
UPDATE on a CDF-enabled table
  data files: rewritten as usual (remove + add)
  change data: extra files under _change_data/ with pre- and post-images of the changed rows
  log: the commit references the change files ("cdc" actions)

INSERT-only commits: no extra files, the inserted rows are read from the added data files
Reading the feed: Delta combines cdc files and added/removed files for the requested versions
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> reading the change feed is a normal (batch or streaming) read.

**Costs:** CDF adds a small amount of write overhead for updates, deletes and merges. Change files are subject to `VACUUM` like data files, so you can only read changes within the retention period.

In [0]:
spark.sql(f"SHOW TBLPROPERTIES {orders}").filter("key = 'delta.enableChangeDataFeed'").show()

In [0]:
for t in [orders, gold]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>DELTA_MISSING_CHANGE_DATA</code> / no changes returned</b><br>
CDF wasn't enabled at that version, or the change files were vacuumed. Enable CDF before you need the changes, and read within the retention period.<br><br>
<b>⛔ Problem: double-counting updates</b><br>You summed both <code>update_preimage</code> and <code>update_postimage</code>. Use post-images (and deletes) for the new state, or subtract pre-images when applying deltas.<br><br>
<b>⛔ Problem: an incremental job reprocesses everything</b><br>It doesn't store the last processed version. Persist it (or use a streaming checkpoint).<br><br>
**⛔ Problem: <code>startingVersion</code> later than the latest version**<br>Nothing new happened since the last run. Handle the empty case.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the change data feed in Delta Lake?</b><br>
A table feature (<code>delta.enableChangeDataFeed</code>) that records row-level changes for each commit: inserts, deletes, and updates as pre- and post-images, with the version and timestamp. You read it with <code>readChangeFeed</code> or <code>table_changes()</code>.<br><br>

<b>🎤 2. What are the values of <code>_change_type</code>?</b><br>
<code>insert</code>, <code>delete</code>, <code>update_preimage</code> (the row before the update) and <code>update_postimage</code> (the row after).<br><br>

<b>🎤 3. How would you use CDF to update a Gold aggregate incrementally?</b><br>
Store the last Silver version processed, read changes since then, find the affected keys, recompute those keys from Silver (or apply deltas), <code>MERGE</code> them into Gold, and save the new version. Or let a streaming query with a checkpoint track the version.<br><br>

<b>🎤 4. CDF vs time travel?</b><br>
Time travel returns a whole table snapshot at a version. CDF returns only the rows that changed between versions, with the type of change, which is what incremental processing needs.<br><br>

<b>🎤 5. Is CDF available for changes made before it was enabled?</b><br>
No, only for commits after it's enabled, and only while the change files are within the retention period.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer needs to process only the rows that were inserted, updated or deleted in a Silver Delta table since version 15. What should they use?</b><br>
A. <code>SELECT * FROM silver VERSION AS OF 15</code><br>
B. <code>SELECT * FROM table_changes('silver', 16)</code> with change data feed enabled<br>
C. <code>DESCRIBE HISTORY silver</code><br>
D. <code>VACUUM silver RETAIN 15 HOURS</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> Time travel returns full snapshots, not the changes.</details><br><br>

<b>Q2. An <code>UPDATE</code> changes one row in a CDF-enabled table. How many rows does the change feed return for it?</b><br>
A. 1 (<code>update</code>)<br>
B. 2 (<code>update_preimage</code> and <code>update_postimage</code>)<br>
C. 3 (<code>delete</code>, <code>insert</code>, <code>update</code>)<br>
D. 0, because updates are not recorded<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a CDF-enabled <code>inventory (sku, qty)</code> table and make 5 changes (inserts, updates, a delete)</li><li>Show the number of changes per <code>_change_type</code></li><li>For one SKU, list its full change history in order</li><li>Compute the net quantity change per SKU between version 1 and the latest version, using post-images minus pre-images (and inserts/deletes)</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
inv = f"{DB}.inventory_cdf"

# 1. CDF table and changes
spark.sql(f"CREATE OR REPLACE TABLE {inv} (sku STRING, qty INT) TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql(f"INSERT INTO {inv} VALUES ('A', 10), ('B', 5)")
spark.sql(f"UPDATE {inv} SET qty = 7 WHERE sku = 'A'")
spark.sql(f"INSERT INTO {inv} VALUES ('C', 3)")
spark.sql(f"UPDATE {inv} SET qty = qty + 1")
spark.sql(f"DELETE FROM {inv} WHERE sku = 'B'")

feed = spark.read.option("readChangeFeed", "true").option("startingVersion", 1).table(inv)

# 2. Changes per type
feed.groupBy("_change_type").count().show()

# 3. History of SKU 'A'
feed.filter("sku = 'A'").orderBy("_commit_version", "_change_type").show()

# 4. Net change: + inserts and post-images, - deletes and pre-images
sign = (F.when(F.col("_change_type").isin("insert", "update_postimage"), 1)
         .when(F.col("_change_type").isin("delete", "update_preimage"), -1))
feed.groupBy("sku").agg(F.sum(sign * F.col("qty")).alias("net_qty_change")).orderBy("sku").show()
spark.sql(f"DROP TABLE {inv}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Enable with <code>delta.enableChangeDataFeed = true</code>. Only later changes are recorded</li><li>Rows carry <code>_change_type</code> (<code>insert</code>, <code>update_preimage</code>, <code>update_postimage</code>, <code>delete</code>), <code>_commit_version</code> and <code>_commit_timestamp</code></li><li>Read with <code>readChangeFeed</code> + <code>startingVersion</code>/<code>endingVersion</code>, or <code>table_changes()</code></li><li>Incremental pattern: changes since the last version → affected keys → recompute → <code>MERGE</code> → save the version</li><li>Streaming with <code>readChangeFeed</code> and a checkpoint tracks progress automatically</li><li>Change files follow <code>VACUUM</code> retention</li></ul>
</div>

| Task | Code |
|---|---|
| Enable | `ALTER TABLE t SET TBLPROPERTIES (delta.enableChangeDataFeed = true)` |
| SQL | `SELECT * FROM table_changes('t', 5)` |
| Python | `spark.read.option("readChangeFeed", "true").option("startingVersion", 5).table("t")` |
| Streaming | `spark.readStream.option("readChangeFeed", "true").table("t")` |

## Git commit

```
git add 03_delta_lake/11_change_data_feed.ipynb
git commit -m "add 03_11 change data feed notebook"
```